In [1]:
# !pip install git+https://github.com/amazon-science/chronos-forecasting.git

In [2]:
import pandas as pd
import numpy as np
import random
from tqdm import tqdm
import matplotlib.pyplot as plt
import torch.nn as nn
import torch
from typing import Union
import warnings
from datetime import datetime
import matplotlib.ticker as ticker
import matplotlib.dates as mdates
from chronos import ChronosPipeline
from Utils import drop_last_n_samples, splitter
import argparse
import sys
import os
import contextlib
import io
import logging
import warnings
import os
import datetime

2025-05-19 15:25:45.915748: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-19 15:25:48.870984: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-05-19 15:26:01.167213: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [3]:

# Function to get the start date of a given year and week
def get_date_from_year_week(year, week):
    try:
        return datetime.datetime.strptime(f'{year}-W{week}-1', "%Y-W%W-%w").date()
    except ValueError:
        # Week 53 issue: Assume last Monday of the year if invalid week
        last_day_of_year = datetime.date(year, 12, 31)
        return last_day_of_year - datetime.timedelta(days=last_day_of_year.weekday())


def prepare_regional_ILI(path_to_data, region):
    # Load data
    data = pd.read_csv(path_to_data)
    data = data[["YEAR", "WEEK", "REGION", "% WEIGHTED ILI"]]
    data.sort_values(by=['REGION', "YEAR", "WEEK"], inplace=True)
    data = data.reset_index(drop=True)

    # Split data to remove noisy part
    _, data = splitter(data, 'REGION', 1000)
    data = data.reset_index(drop=True)

    # Add Indicator instead of date
    data["Indicator"] = data.groupby("REGION").cumcount() + 1

    # Assuming your DataFrame is named ili_df
    data["DATE"] = data.apply(lambda row: get_date_from_year_week(row["YEAR"], row["WEEK"]), axis=1)

    # Prepare the final DataFrame
    df = data[["Indicator", "REGION", "% WEIGHTED ILI"]]
    df.columns = ["ds", "unique_id", "y"]
    df.loc[:, 'y'] = df['y'].astype(float)

    Using_validation = "No"
    number_of_time_series = len(df['unique_id'].unique())
    length_time_series = int(len(df) / len(df['unique_id'].unique()))
    forecasting_horizon= 1

    # Identify unique time series
    unique_ids = df['unique_id'].unique()
    Cross_Validation = 5
    train_ids = unique_ids[unique_ids != region]
    test_ids = [region]

    # Create train and test DataFrames
    train_df = df[df['unique_id'].isin(train_ids)]
    test_df = df[df['unique_id'].isin(test_ids)]

    return test_df, data


In [6]:
import io, os, contextlib, warnings, logging
import pandas as pd
import numpy as np
import torch, pytorch_lightning as pl
from tqdm.auto import tqdm
from chronos import ChronosPipeline, BaseChronosPipeline        # or `from amazon_chronos import …`
import matplotlib.pyplot as plt                   # only needed if you keep the loss plot

# ────────────────────────────────────────────────────────────────────────────────
def Chronos_evaluate(
        path_to_data: str,
        unique_ids: list[str],
        core: str,
        number_of_time_series: int,
        length_time_series: int,
        forecasting_horizon: int,                 # ❶ <- set to 4 when you call the fn
        number_of_samples_in_test: int,
        seed: int,
        look_back_size: int,
        iteration_number: int,
        path_to_results: str,
        ):
    """Evaluate Chronos on 4-step forecasts and log h1-h4 metrics + avg MSE."""
    pl.seed_everything(seed, workers=True)
    model_name   = "Chronos"
    chronos_pipe = BaseChronosPipeline.from_pretrained(
                    f"amazon/{core}",
                    device_map="cuda",
                    torch_dtype=torch.bfloat16,
                    )
    
    def transform_df(df):
        first_ds_values = df.groupby("unique_id")["ds"].first().reset_index()
        first_ds_values.rename(columns={"ds": "first_ds"}, inplace=True)
        df_pivot = df.pivot(index='unique_id', columns='ds', values=model)
        df_pivot.columns = ['y_hat_1', 'y_hat_2', 'y_hat_3', 'y_hat_4']
        df_pivot = df_pivot.reset_index()
        df_pivot = df_pivot.merge(first_ds_values, on="unique_id")
        column_order = ["first_ds", "unique_id", 'y_hat_1', 'y_hat_2', 'y_hat_3', 'y_hat_4']
        df_pivot = df_pivot[column_order]
        return df_pivot

    
    
    
    test_df = pd.DataFrame(columns=["first_ds", "unique_id", "y"])
    print("Starting evaluation ....")
    all_preds = pd.DataFrame(columns=["first_ds", "unique_id", "Chronos"])
    for region in unique_ids:
        print("Evaluating",region)
        df, data = prepare_regional_ILI(path_to_data, region)
        _, test_part_df = splitter(df, 'unique_id', number_of_samples_in_test)
        test_df = pd.concat([test_df, test_part_df], ignore_index=True)
        for idx, i in enumerate(range(number_of_samples_in_test, 0, -1)):
            with contextlib.redirect_stdout(io.StringIO()):
                _, data_batch_part = splitter(df, 'unique_id', i + look_back_size)
                data_batch_part = drop_last_n_samples(data_batch_part, 'unique_id', i)

            # Convert context to PyTorch tensor
            context_tensor = torch.tensor(data_batch_part["y"].values[-look_back_size:], dtype=torch.float32)

            preds = chronos_pipe.predict(
                context=context_tensor,
                prediction_length=forecasting_horizon
            )
            preds = preds.squeeze(0) 
            h1, h2, h3, h4 = preds.mean(dim=0)  

            # Append to DataFrame
            all_preds = pd.concat([all_preds, pd.DataFrame({"first_ds": [idx+801], "unique_id": region, "y_hat_1": h1.item(), "y_hat_2":h2.item(), "y_hat_3":h3.item(), "y_hat_4":h4.item()})], ignore_index=True)
            
    
    def shift_up_by_group(df, group_col, columns_to_shift, shift_by, fill_value=0):
        shifted_groups = []
        unique_groups = df[group_col].unique()
        for grp_value in unique_groups:
            group_data = df[df[group_col] == grp_value].copy()
            group_data[columns_to_shift] = group_data[columns_to_shift].shift(shift_by)
            shifted_groups.append(group_data)
        df_shifted = pd.concat(shifted_groups, ignore_index=True)
        return df_shifted[columns_to_shift]


    all_preds = all_preds.sort_values(by=['unique_id', 'first_ds'], ascending=[True, True])
    test_df = test_df.sort_values(by=['unique_id', 'ds'], ascending=[True, True])
    results_df = all_preds.copy()
    results_df = results_df.reset_index(drop=True)
    test_df    = test_df.reset_index(drop=True)
    results_df['y_actual_1'] = test_df['y'].values
    results_df['y_actual_2'] = shift_up_by_group(test_df, group_col='unique_id', columns_to_shift=['y'],shift_by=-1, fill_value=0)
    results_df['y_actual_3'] = shift_up_by_group(test_df, group_col='unique_id', columns_to_shift=['y'],shift_by=-2, fill_value=0)
    results_df['y_actual_4'] = shift_up_by_group(test_df, group_col='unique_id', columns_to_shift=['y'],shift_by=-3, fill_value=0)


    def results_calculator(df, group_col, traget_y, predicted_y, shift_by, number_of_time_series, number_of_samples_in_test):
        filtered_df = df[['first_ds', 'unique_id', traget_y, predicted_y]]
        shifted_groups = []
        unique_groups = df[group_col].unique()
        for grp_value in unique_groups:
            group_data = df[df[group_col] == grp_value].copy()
            group_data.drop(group_data.tail(shift_by).index, inplace=True)
            shifted_groups.append(group_data)
        df_shifted = pd.concat(shifted_groups, ignore_index=True)

        Y_ACTUAL = df_shifted[traget_y].values.reshape(number_of_time_series, number_of_samples_in_test-shift_by)
        Y_HAT = df_shifted[predicted_y].values.reshape(number_of_time_series, number_of_samples_in_test-shift_by)

        mae = np.mean(np.abs(Y_ACTUAL - Y_HAT))
        mse = np.mean(np.square(Y_ACTUAL - Y_HAT))
        nnse = 1 / (2 - (1 - np.sum(np.square(Y_ACTUAL - Y_HAT)) / np.sum(np.square(Y_ACTUAL - np.mean(Y_ACTUAL)))))
        r2 = 1 - (np.sum(np.square(Y_ACTUAL - Y_HAT)) / np.sum(np.square(Y_ACTUAL - np.mean(Y_ACTUAL))))
        smape = 100 * np.mean(2 * np.abs(Y_HAT - Y_ACTUAL) / (np.abs(Y_ACTUAL) + np.abs(Y_HAT) + 1e-8))

        return mae, mse, nnse, smape, r2


    mae_1, mse_1, nnse_1, smape_1, r2_1 = results_calculator(results_df, 'unique_id', 'y_actual_1', 'y_hat_1', 0, number_of_time_series, number_of_samples_in_test)
    mae_2, mse_2, nnse_2, smape_2, r2_2 = results_calculator(results_df, 'unique_id', 'y_actual_2', 'y_hat_2', 1, number_of_time_series, number_of_samples_in_test)
    mae_3, mse_3, nnse_3, smape_3, r2_3 = results_calculator(results_df, 'unique_id', 'y_actual_3', 'y_hat_3', 2, number_of_time_series, number_of_samples_in_test)
    mae_4, mse_4, nnse_4, smape_4, r2_4 = results_calculator(results_df, 'unique_id', 'y_actual_4', 'y_hat_4', 3, number_of_time_series, number_of_samples_in_test)

    avg_mse = (mse_1+mse_2+mse_3+mse_4)/4

    # Save results
    config_text = f"""\
    Dataset: ILI
    Model: {model}
    Target_series: % WEIGHTED ILI
    Feature: % WEIGHTED ILI
    Normalization: None

    Splitting_strategy: Time-based
    Validation: No
    Samples in one time series: {length_time_series}
    Samples in test part: {number_of_samples_in_test}
    Number of time series: {number_of_time_series}
    Frequency: '1W'

    forecasting_horizon: {forecasting_horizon}
    Look back size: {look_back_size}
    seed_neuralforecast: {seed}

    ***RESULTS h1***
    MSE:    {mse_1:.5f}
    MAE:    {mae_1:.5f}
    SMAPE:  {smape_1:.5f}
    NNSE:   {nnse_1:.5f}
    R²:     {r2_1:.5f}

    ***RESULTS h2***
    MSE:    {mse_2:.5f}
    MAE:    {mae_2:.5f}
    SMAPE:  {smape_2:.5f}
    NNSE:   {nnse_2:.5f}
    R²:     {r2_2:.5f}

    ***RESULTS h3***
    MSE:    {mse_3:.5f}
    MAE:    {mae_3:.5f}
    SMAPE:  {smape_3:.5f}
    NNSE:   {nnse_3:.5f}
    R²:     {r2_3:.5f}

    ***RESULTS h4***
    MSE:    {mse_4:.5f}
    MAE:    {mae_4:.5f}
    SMAPE:  {smape_4:.5f}
    NNSE:   {nnse_4:.5f}
    R²:     {r2_4:.5f}

    ***Average MSE***
    avg_mse: {avg_mse}
    """

    with open(f"{path_to_results}/Results_TB_h{forecasting_horizon}_s{seed}.txt", "w") as file:
        file.write(config_text)

    print(config_text)


In [19]:

# ───────────────────── experiment settings ─────────────────────
model                     = "Chronos"
core                      = "chronos-bolt-mini"
path_to_data              = "ILINet.csv"

forecasting_horizon       = 4        
look_back_size            = 52
iteration_number          = 20
number_of_time_series     = 10
length_time_series        = 1000
number_of_samples_in_test = 200

unique_ids = [
    "Region 1", "Region 2", "Region 3", "Region 4", "Region 5",
    "Region 6", "Region 7", "Region 8", "Region 9", "Region 10"
]

# ─────────────────────────── run seeds ─────────────────────────
for seed in range(1, 11):
    path_to_results = f"{model}_core_{core}/TB_h4_lbs_{look_back_size}/seed_{seed}"
    os.makedirs(path_to_results, exist_ok=True)

    Chronos_evaluate(
        path_to_data=path_to_data,
        unique_ids=unique_ids,
        core=core,
        number_of_time_series=number_of_time_series,
        length_time_series=length_time_series,
        forecasting_horizon=forecasting_horizon,
        number_of_samples_in_test=number_of_samples_in_test,
        seed=seed,
        look_back_size=look_back_size,
        iteration_number=iteration_number,
        path_to_results=path_to_results,
    )


[rank: 0] Seed set to 1


Starting evaluation ....
Evaluating Region 1
Evaluating Region 2
Evaluating Region 3
Evaluating Region 4
Evaluating Region 5
Evaluating Region 6
Evaluating Region 7
Evaluating Region 8
Evaluating Region 9
Evaluating Region 10


[rank: 0] Seed set to 2


    Dataset: ILI
    Model: Chronos
    Target_series: % WEIGHTED ILI
    Feature: % WEIGHTED ILI
    Normalization: None

    Splitting_strategy: Time-based
    Validation: No
    Samples in one time series: 1000
    Samples in test part: 200
    Number of time series: 10
    Frequency: '1W'

    forecasting_horizon: 4
    Look back size: 52
    seed_neuralforecast: 1

    ***RESULTS h1***
    MSE:    0.20077
    MAE:    0.25352
    SMAPE:  9.72383
    NNSE:   0.92236
    R²:     0.91583

    ***RESULTS h2***
    MSE:    0.56086
    MAE:    0.42096
    SMAPE:  15.60244
    NNSE:   0.80960
    R²:     0.76482

    ***RESULTS h3***
    MSE:    0.93886
    MAE:    0.56779
    SMAPE:  21.11937
    NNSE:   0.71754
    R²:     0.60634

    ***RESULTS h4***
    MSE:    1.35499
    MAE:    0.70554
    SMAPE:  26.23851
    NNSE:   0.63775
    R²:     0.43199

    ***Average MSE***
    avg_mse: 0.7638680748778214
    
Starting evaluation ....
Evaluating Region 1
Evaluating Region 2
Evaluating R

[rank: 0] Seed set to 3


...

    Dataset: ILI
    Model: Chronos
    Target_series: % WEIGHTED ILI
    Feature: % WEIGHTED ILI
    Normalization: None

    Splitting_strategy: Time-based
    Validation: No
    Samples in one time series: 1000
    Samples in test part: 200
    Number of time series: 10
    Frequency: '1W'

    forecasting_horizon: 4
    Look back size: 52
    seed_neuralforecast: 7

    ***RESULTS h1***
    MSE:    0.20077
    MAE:    0.25352
    SMAPE:  9.72383
    NNSE:   0.92236
    R²:     0.91583

    ***RESULTS h2***
    MSE:    0.56086
    MAE:    0.42096
    SMAPE:  15.60244
    NNSE:   0.80960
    R²:     0.76482

    ***RESULTS h3***
    MSE:    0.93886
    MAE:    0.56779
    SMAPE:  21.11937
    NNSE:   0.71754
    R²:     0.60634

    ***RESULTS h4***
    MSE:    1.35499
    MAE:    0.70554
    SMAPE:  26.23851
    NNSE:   0.63775
    R²:     0.43199

    ***Average MSE***
    avg_mse: 0.7638680748778214
    
Starting evaluation ....
Evaluating Region 1
Evaluating Region 2
Evaluating R

[rank: 0] Seed set to 9


    Dataset: ILI
    Model: Chronos
    Target_series: % WEIGHTED ILI
    Feature: % WEIGHTED ILI
    Normalization: None

    Splitting_strategy: Time-based
    Validation: No
    Samples in one time series: 1000
    Samples in test part: 200
    Number of time series: 10
    Frequency: '1W'

    forecasting_horizon: 4
    Look back size: 52
    seed_neuralforecast: 8

    ***RESULTS h1***
    MSE:    0.20077
    MAE:    0.25352
    SMAPE:  9.72383
    NNSE:   0.92236
    R²:     0.91583

    ***RESULTS h2***
    MSE:    0.56086
    MAE:    0.42096
    SMAPE:  15.60244
    NNSE:   0.80960
    R²:     0.76482

    ***RESULTS h3***
    MSE:    0.93886
    MAE:    0.56779
    SMAPE:  21.11937
    NNSE:   0.71754
    R²:     0.60634

    ***RESULTS h4***
    MSE:    1.35499
    MAE:    0.70554
    SMAPE:  26.23851
    NNSE:   0.63775
    R²:     0.43199

    ***Average MSE***
    avg_mse: 0.7638680748778214
    
Starting evaluation ....
Evaluating Region 1
Evaluating Region 2
Evaluating R

[rank: 0] Seed set to 10


    Dataset: ILI
    Model: Chronos
    Target_series: % WEIGHTED ILI
    Feature: % WEIGHTED ILI
    Normalization: None

    Splitting_strategy: Time-based
    Validation: No
    Samples in one time series: 1000
    Samples in test part: 200
    Number of time series: 10
    Frequency: '1W'

    forecasting_horizon: 4
    Look back size: 52
    seed_neuralforecast: 9

    ***RESULTS h1***
    MSE:    0.20077
    MAE:    0.25352
    SMAPE:  9.72383
    NNSE:   0.92236
    R²:     0.91583

    ***RESULTS h2***
    MSE:    0.56086
    MAE:    0.42096
    SMAPE:  15.60244
    NNSE:   0.80960
    R²:     0.76482

    ***RESULTS h3***
    MSE:    0.93886
    MAE:    0.56779
    SMAPE:  21.11937
    NNSE:   0.71754
    R²:     0.60634

    ***RESULTS h4***
    MSE:    1.35499
    MAE:    0.70554
    SMAPE:  26.23851
    NNSE:   0.63775
    R²:     0.43199

    ***Average MSE***
    avg_mse: 0.7638680748778214
    
Starting evaluation ....
Evaluating Region 1
Evaluating Region 2
Evaluating R

In [26]:
import os
import glob
import statistics
import re

def parse_metrics_from_file(filepath):
    """
    Parse the desired metrics from a single results file.
    Returns a dictionary with the following structure:
    {
      "training_loss": float,
      "avg_mse": float,
      "h1": {"MSE": float, "MAE": float, "SMAPE": float, "NNSE": float, "R2": float},
      "h2": {...},
      "h3": {...},
      "h4": {...}
    }
    If something is not found, returns None for that field.
    """

    # Initialize the dict with None (or any default) for each metric
    metrics_dict = {
        "avg_mse": None,
        "h1": {"MSE": None, "MAE": None, "SMAPE": None, "NNSE": None, "R2": None},
        "h2": {"MSE": None, "MAE": None, "SMAPE": None, "NNSE": None, "R2": None},
        "h3": {"MSE": None, "MAE": None, "SMAPE": None, "NNSE": None, "R2": None},
        "h4": {"MSE": None, "MAE": None, "SMAPE": None, "NNSE": None, "R2": None},
    }

    # Regex patterns to extract numbers (floats). 
    # We'll look for lines like: "training loss (MSE): 0.39443" or "MSE:    0.15226"
    # Adjust the patterns if your file format changes.
    pattern_training_loss = re.compile(r"training loss.*:\s*([0-9.]+)")
    pattern_avg_mse = re.compile(r"avg_mse:\s*([0-9.]+)")

    # For lines like "***RESULTS h1***" we capture h1
    # For lines like "MSE:\s*([0-9.]+)"
    pattern_horizon = re.compile(r"\*\*\*RESULTS\s+(h[0-9]+)\*\*\*")
    pattern_key_value = re.compile(r"(MSE|MAE|SMAPE|NNSE|R²?):\s*([0-9.]+)")

    current_horizon = None

    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()

            # 1) Training Loss
            train_match = pattern_training_loss.search(line)
            if train_match:
                metrics_dict["training_loss"] = float(train_match.group(1))
                continue

            # 2) Average MSE
            avg_mse_match = pattern_avg_mse.search(line)
            if avg_mse_match:
                metrics_dict["avg_mse"] = float(avg_mse_match.group(1))
                continue

            # 3) Detect horizon header (e.g. ***RESULTS h1***)
            horizon_match = pattern_horizon.search(line)
            if horizon_match:
                current_horizon = horizon_match.group(1)  # e.g. 'h1', 'h2', etc.
                continue

            # 4) Inside horizon block, parse lines like "MSE:    0.15226"
            if current_horizon is not None:
                kv_match = pattern_key_value.search(line)
                if kv_match:
                    metric_name = kv_match.group(1)  # e.g. 'MSE'
                    metric_value = float(kv_match.group(2))

                    # Correct potential mismatch 'R²' vs 'R2' dictionary key
                    if metric_name.startswith('R'):
                        metric_name = "R2"

                    metrics_dict[current_horizon][metric_name] = metric_value

    return metrics_dict


def main(results_folder="path/to/results_folder"):
    """
    1. Collect all .txt files (or any extension) from the specified results_folder
    2. Parse metrics for each file
    3. Aggregate all metrics (mean, std)
    """

    # A structure to accumulate metric values across all files
    # We'll store lists, then compute mean and std afterwards
    all_metrics = {
        "training_loss": [],
        "avg_mse": [],
        "h1": {"MSE": [], "MAE": [], "SMAPE": [], "NNSE": [], "R2": []},
        "h2": {"MSE": [], "MAE": [], "SMAPE": [], "NNSE": [], "R2": []},
        "h3": {"MSE": [], "MAE": [], "SMAPE": [], "NNSE": [], "R2": []},
        "h4": {"MSE": [], "MAE": [], "SMAPE": [], "NNSE": [], "R2": []},
    }

    # Modify the glob pattern as needed (e.g. *.txt, *.log, etc.)
    file_list = glob.glob(os.path.join(results_folder, "*.txt"))

    for file_path in file_list:
        parsed = parse_metrics_from_file(file_path)

        if parsed["avg_mse"] is not None:
            all_metrics["avg_mse"].append(parsed["avg_mse"])

        for horizon in ["h1", "h2", "h3", "h4"]:
            for mkey in ["MSE", "MAE", "SMAPE", "NNSE", "R2"]:
                val = parsed[horizon].get(mkey, None)
                if val is not None:
                    all_metrics[horizon][mkey].append(val)

    # Helper to compute mean and sample std
    def mean_std(values):
        if len(values) == 0:
            return (None, None)
        return (statistics.mean(values), statistics.stdev(values))

    # Print aggregated results
    print("===== AGGREGATED RESULTS ACROSS ALL FILES =====")

    # avg_mse
    mean_val, std_val = mean_std(all_metrics["avg_mse"])
    print(f"Average MSE:         mean={mean_val:.3f}, std={std_val:.3f}")

    # For each horizon and each metric
    for horizon in ["h1", "h2", "h3", "h4"]:
        print(f"\n--- {horizon.upper()} ---")
        for metric_name in ["MSE", "MAE", "SMAPE", "NNSE", "R2"]:
            mean_val, std_val = mean_std(all_metrics[horizon][metric_name])
            print(f"{metric_name}: mean={mean_val:.3f}, std={std_val:.3f}")


     
print(path_to_results)
path_to_results = f"{model}_core_{core}/TB_h4_lbs_{look_back_size}"
main(results_folder=path_to_results)




Chronos_core_chronos-bolt-small/TB_h4_lbs_52
===== AGGREGATED RESULTS ACROSS ALL FILES =====
Average MSE:         mean=0.747, std=0.000

--- H1 ---
MSE: mean=0.193, std=0.000
MAE: mean=0.254, std=0.000
SMAPE: mean=9.725, std=0.000
NNSE: mean=0.925, std=0.000
R2: mean=0.919, std=0.000

--- H2 ---
MSE: mean=0.560, std=0.000
MAE: mean=0.429, std=0.000
SMAPE: mean=15.850, std=0.000
NNSE: mean=0.810, std=0.000
R2: mean=0.765, std=0.000

--- H3 ---
MSE: mean=0.917, std=0.000
MAE: mean=0.574, std=0.000
SMAPE: mean=21.342, std=0.000
NNSE: mean=0.722, std=0.000
R2: mean=0.615, std=0.000

--- H4 ---
MSE: mean=1.317, std=0.000
MAE: mean=0.713, std=0.000
SMAPE: mean=26.504, std=0.000
NNSE: mean=0.644, std=0.000
R2: mean=0.448, std=0.000


### Chronos-Bolt-mini
===== AGGREGATED RESULTS ACROSS ALL FILES =====

Average MSE:         mean=0.764, std=0.000

--- H1 ---
MSE: mean=0.201, std=0.000
NNSE: mean=0.922, std=0.000

--- H2 ---
MSE: mean=0.561, std=0.000
NNSE: mean=0.810, std=0.000

--- H3 ---
MSE: mean=0.939, std=0.000
NNSE: mean=0.718, std=0.000

--- H4 ---
MSE: mean=1.355, std=0.000
NNSE: mean=0.638, std=0.000



### Chronos-Bolt-base
===== AGGREGATED RESULTS ACROSS ALL FILES =====

Average MSE:         mean=0.730, std=0.000

--- H1 ---
MSE: mean=0.183, std=0.000
NNSE: mean=0.929, std=0.000

--- H2 ---
MSE: mean=0.534, std=0.000
NNSE: mean=0.817, std=0.000

--- H3 ---
MSE: mean=0.898, std=0.000
NNSE: mean=0.726, std=0.000

--- H4 ---
MSE: mean=1.305, std=0.000
NNSE: mean=0.646, std=0.000



### Chronos-Bolt-small
===== AGGREGATED RESULTS ACROSS ALL FILES =====

Average MSE:         mean=0.747, std=0.000

--- H1 ---
MSE: mean=0.193, std=0.000
NNSE: mean=0.925, std=0.000

--- H2 ---
MSE: mean=0.560, std=0.000
NNSE: mean=0.810, std=0.000

--- H3 ---
MSE: mean=0.917, std=0.000
NNSE: mean=0.722, std=0.000

--- H4 ---
MSE: mean=1.317, std=0.000
NNSE: mean=0.644, std=0.000